<a href="https://colab.research.google.com/github/UH-HACKATHON-TEAMN/Hackathon-Town-C/blob/main/ramya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Styling plots
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load training and testing datasets[cite: 1]
train_df = pd.read_csv('dataset_C_training.csv')
test_df = pd.read_csv('dataset_C_testing.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")

# Check missing values in training data
missing_counts = train_df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]
print("\nFeatures with missing values:")
display(missing_counts)

In [ ]:
# Plot 1: Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='covid_vaccine', data=train_df, palette='Set2')
plt.title('Distribution of Covid Vaccine (Target Variable)')
plt.xlabel('Covid Vaccine (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Plot 2: Doctor Recommendation vs Vaccine Uptake
plt.figure(figsize=(8, 5))
sns.countplot(x='doctor_recc_covid', hue='covid_vaccine', data=train_df, palette='Set1')
plt.title('Covid Vaccine Uptake by Doctor Recommendation')
plt.xlabel('Doctor Recommendation (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.legend(title='Got Vaccine', labels=['No', 'Yes'])
plt.show()

# Plot 3: Age Group vs Vaccine Uptake
plt.figure(figsize=(10, 5))
sns.countplot(y='age_group', hue='covid_vaccine', data=train_df, palette='coolwarm', order=train_df['age_group'].value_counts().index)
plt.title('Vaccine Uptake Across Age Groups')
plt.xlabel('Count')
plt.ylabel('Age Group')
plt.legend(title='Got Vaccine', labels=['No', 'Yes'])
plt.show()

In [ ]:
# Separate features and target variable
X = train_df.drop(columns=['respondent_id', 'covid_vaccine'])
y = train_df['covid_vaccine']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Define preprocessing pipelines for numerical and categorical features
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# Split into train and validation sets with stratification
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training split: {X_train.shape}, Validation split: {X_val.shape}")

In [ ]:
# Build and train Random Forest Pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42))
])

rf_pipeline.fit(X_train, y_train)

# Generate validation probabilities
val_probabilities = rf_pipeline.predict_proba(X_val)[:, 1]
val_roc_auc = roc_auc_score(y_val, val_probabilities)
print(f"Validation ROC-AUC Score: {val_roc_auc:.4f}")

In [ ]:
# Business cost parameters from project brief[cite: 3]
VACCINE_COST = 1500
HOSP_COST = 41000
HOSP_PROB_VAX = 0.02
HOSP_PROB_UNVAX = 0.20

def calculate_total_town_cost(probs, threshold):
    preds = (probs >= threshold).astype(int)
    num_vax = np.sum(preds == 1)
    num_unvax = np.sum(preds == 0)

    total_vaccine_spend = num_vax * VACCINE_COST
    expected_hosp_spend = (num_vax * HOSP_PROB_VAX + num_unvax * HOSP_PROB_UNVAX) * HOSP_COST
    return total_vaccine_spend + expected_hosp_spend

# Search for the decision threshold that minimizes total town expenditure
best_threshold = 0.5
min_town_cost = float('inf')
thresholds = np.linspace(0.0, 1.0, 101)
cost_history = []

for th in thresholds:
    cost = calculate_total_town_cost(val_probabilities, th)
    cost_history.append(cost)
    if cost < min_town_cost:
        min_town_cost = cost
        best_threshold = th

print(f"Optimal Decision Threshold: {best_threshold:.2f}")
print(f"Minimum Optimized Town Cost: ${min_town_cost:,.2f}")

# Plot Cost Curve
plt.figure(figsize=(8, 4))
plt.plot(thresholds, cost_history, color='purple', lw=2)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Optimal Threshold ({best_threshold:.2f})')
plt.title('Town Total Cost vs. Decision Threshold')
plt.xlabel('Prediction Threshold')
plt.ylabel('Total Town Cost ($)')
plt.legend()
plt.show()

In [ ]:
# Business cost parameters from project brief[cite: 3]
VACCINE_COST = 1500
HOSP_COST = 41000
HOSP_PROB_VAX = 0.02
HOSP_PROB_UNVAX = 0.20

def calculate_total_town_cost(probs, threshold):
    preds = (probs >= threshold).astype(int)
    num_vax = np.sum(preds == 1)
    num_unvax = np.sum(preds == 0)

    total_vaccine_spend = num_vax * VACCINE_COST
    expected_hosp_spend = (num_vax * HOSP_PROB_VAX + num_unvax * HOSP_PROB_UNVAX) * HOSP_COST
    return total_vaccine_spend + expected_hosp_spend

# Search for the decision threshold that minimizes total town expenditure
best_threshold = 0.5
min_town_cost = float('inf')
thresholds = np.linspace(0.0, 1.0, 101)
cost_history = []

for th in thresholds:
    cost = calculate_total_town_cost(val_probabilities, th)
    cost_history.append(cost)
    if cost < min_town_cost:
        min_town_cost = cost
        best_threshold = th

print(f"Optimal Decision Threshold: {best_threshold:.2f}")
print(f"Minimum Optimized Town Cost: ${min_town_cost:,.2f}")

# Plot Cost Curve
plt.figure(figsize=(8, 4))
plt.plot(thresholds, cost_history, color='purple', lw=2)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Optimal Threshold ({best_threshold:.2f})')
plt.title('Town Total Cost vs. Decision Threshold')
plt.xlabel('Prediction Threshold')
plt.ylabel('Total Town Cost ($)')
plt.legend()
plt.show()

In [ ]:
from sklearn.svm import SVC

# Define and train SVM Pipeline
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, C=1, kernel='rbf', random_state=42))
])

print("Training Support Vector Machine...")
svm_pipeline.fit(X_train, y_train)

# Generate validation probabilities for SVM
svm_val_probabilities = svm_pipeline.predict_proba(X_val)[:, 1]
svm_val_roc_auc = roc_auc_score(y_val, svm_val_probabilities)
print(f"Validation ROC-AUC Score (SVM): {svm_val_roc_auc:.4f}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Define and train Gradient Boosting / XGBoost Pipeline
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42))
])

print("Training XGBoost / Gradient Boosting...")
xgb_pipeline.fit(X_train, y_train)

# Generate validation probabilities for XGBoost
xgb_val_probabilities = xgb_pipeline.predict_proba(X_val)[:, 1]
xgb_val_roc_auc = roc_auc_score(y_val, xgb_val_probabilities)
print(f"Validation ROC-AUC Score (XGBoost): {xgb_val_roc_auc:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter grids for each model
rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

svm_param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
    'classifier__kernel': ['linear', 'rbf']
}

gb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    'classifier__subsample': [0.8, 1.0]
}

# Dictionary of models and their respective tuning grids
models_to_tune = {
    "Random Forest": (RandomForestClassifier(random_state=42), rf_param_grid),
    "Support Vector Machine": (SVC(probability=True, random_state=42), svm_param_grid),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42), gb_param_grid)
}

tuned_pipelines = {}
tuning_results = []

print("=== STARTING HYPERPARAMETER TUNING (RandomizedSearchCV) ===")

for name, (model, param_grid) in models_to_tune.items():
    print(f"\nTuning {name}...")

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Randomized Search with 3-fold Cross-Validation optimizing for ROC-AUC
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=5,             # Number of parameter settings sampled
        cv=3,                 # 3-fold cross-validation
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    best_estimator = search.best_estimator_
    tuned_pipelines[name] = best_estimator

    # Evaluate on validation set
    val_probs = best_estimator.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)

    # Cost optimization for tuned model
    best_th = 0.5
    min_cost = float('inf')
    for th in np.linspace(0.0, 1.0, 101):
        preds = (val_probs >= th).astype(int)
        num_vax = np.sum(preds == 1)
        num_unvax = np.sum(preds == 0)
        cost = (num_vax * 1500) + ((num_vax * 0.02 + num_unvax * 0.20) * 41000)
        if cost < min_cost:
            min_cost = cost
            best_th = th

    tuning_results.append({
        'Model': name,
        'Best Hyperparameters': search.best_params_,
        'Validation ROC-AUC': val_auc,
        'Optimal Threshold': best_th,
        'Minimized Town Cost ($)': min_cost
    })
    print(f"Best {name} Params: {search.best_params_}")
    print(f"Minimized Town Cost: ${min_cost:,.2f}")

# Display comparison of tuned models
tuned_comparison_df = pd.DataFrame(tuning_results).sort_values(by='Minimized Town Cost ($)', ascending=True)
print("\n=== TUNED MODEL COMPARISON SUMMARY ===")
display(tuned_comparison_df[['Model', 'Validation ROC-AUC', 'Optimal Threshold', 'Minimized Town Cost ($)']])

=== STARTING HYPERPARAMETER TUNING (RandomizedSearchCV) ===

Tuning Random Forest...
Best Random Forest Params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 30}
Minimized Town Cost: $2,208,640.00

Tuning Support Vector Machine...


In [ ]:
# Gather all model probabilities
model_evals = {
    "Random Forest": val_probabilities,
    "Support Vector Machine": svm_val_probabilities,
    "XGBoost / Gradient Boosting": xgb_val_probabilities
}

comparison_data = []

print("=== Evaluating and Optimizing Costs for All Models ===")
for name, probs in model_evals.items():
    auc = roc_auc_score(y_val, probs)

    # Find optimal threshold for this model
    best_th = 0.5
    min_cost = float('inf')

    for th in np.linspace(0.0, 1.0, 101):
        preds = (probs >= th).astype(int)
        num_vax = np.sum(preds == 1)
        num_unvax = np.sum(preds == 0)

        # Business cost calculation ($1,500 vaccine vs $41,000 hospitalization)[cite: 3]
        cost = (num_vax * 1500) + ((num_vax * 0.02 + num_unvax * 0.20) * 41000)
        if cost < min_cost:
            min_cost = cost
            best_th = th

    comparison_data.append({
        'Model': name,
        'ROC-AUC Score': auc,
        'Optimal Threshold': best_th,
        'Minimized Town Cost ($)': min_cost
    })

comparison_df = pd.DataFrame(comparison_data).sort_values(by='Minimized Town Cost ($)', ascending=True)
print("\n=== MODEL COMPARISON RESULTS ===")
display(comparison_df)

In [ ]:
# Automatically select the best model based on minimum town cost
best_row = comparison_df.iloc[0]
best_model_name = best_row['Model']
best_threshold = best_row['Optimal Threshold']

print(f"*** Winning Model: {best_model_name} (Threshold: {best_threshold:.2f}) ***")

# Map model name back to its pipeline
pipelines_dict = {
    "Random Forest": rf_pipeline,
    "Support Vector Machine": svm_pipeline,
    "XGBoost / Gradient Boosting": xgb_pipeline
}
winning_pipeline = pipelines_dict[best_model_name]

# Predict on test dataset[cite: 3]
X_test = test_df.drop(columns=['respondent_id'])
test_probs = winning_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

# Create final submission dataframe[cite: 3]
final_submission = pd.DataFrame({
    'respondent_id': test_df['respondent_id'],
    'covid_vaccine_prediction': test_preds,
    'covid_vaccine_probability': test_probs
})

output_filename = 'dataset_C_best_model_submission.csv'
final_submission.to_csv(output_filename, index=False)
print(f"Submission file saved successfully as '{output_filename}'!")
display(final_submission.head(10))